In [ ]:
from app.models.statements import Comment, ConversationStarter
from app.i18n import Topic

In [ ]:
Comment(text="hello", reply_to=None, topics=("protest",)).topics

---
## figuring out queueing of comments

In [ ]:
# from app.db import seed_DB
import random

def seed_DB() -> dict[int, Statement]:
    starter = ConversationStarter(text="i think", topics=("what_is_democracy",))
    init_comments = [
        Comment(text="hi", reply_to=starter.ID, topics=None),
        Comment(text="bye", reply_to=None, topics=("protest",)),
    ]
    return [starter], {c.ID: c for c in init_comments}

starters, DB = seed_DB()
# starters = [c for c in DB.values() if isinstance(c, ConversationStarter)]

In [ ]:
def start_conversation():
    return random.choice(starters)

def next_comment(cur, emitted, max_length=5):
    if emitted >= max_length:
        return start_conversation()
    else:
        return random.choice(DB.values())

def random_topics():
    k = random.randint(1, len(list(Topic))//2)
    return random.sample(list(Topic), k)



def maybe_new_comment(comment_counter):
    if random.random() < 0.2:
        if random.random() < 0.3:
            new_comment = Comment(text=f"new_{comment_counter}", reply_to=None,
                                  topics=random_topics())
        else:
            new_comment = Comment(text=f"new_{comment_counter}", reply_to=cur.ID,
                                  topics=None)
        comment_counter += 1
        return new_comment
    return None
    

cur = start_conversation()
collected = [cur]
least_length = 4
comment_counter = 1

for _ in range(20):
    new_comment = maybe_new_comment(comment_counter)
    print(new_comment)

In [ ]:
import asyncio
import random

async def ticker(queue, interval=2.0):
    n = 0
    while True:
        await asyncio.sleep(interval)
        n += 1
        await queue.put(f"tick {n}")

async def interceptor(queue):
    while True:
        await asyncio.sleep(random.uniform(0.5, 5.0))
        if random.random() < 0.4:
            await queue.put("INTERCEPT: new comment arrived")

async def consumer(queue, duration=20):
    loop = asyncio.get_running_loop()
    end = loop.time() + duration
    while loop.time() < end:
        try:
            event = await asyncio.wait_for(queue.get(), timeout=end - loop.time())
        except asyncio.TimeoutError:
            break
        print(f"{loop.time():8.2f}  {event}")

async def main(duration=200):
    queue = asyncio.Queue()
    tasks = [asyncio.create_task(ticker(queue)),
             asyncio.create_task(interceptor(queue))]
    try:
        await consumer(queue, duration)
    finally:
        for t in tasks:
            t.cancel()
        await asyncio.gather(*tasks, return_exceptions=True)

await main()   # in a script, use asyncio.run(main()) instead

In [ ]:
import asyncio
import random

async def interceptor(queue):
    while True:
        await asyncio.sleep(random.uniform(0.3, 4.0))
        if random.random() < 0.5:
            await queue.put("new comment arrived")

async def ticker(queue, interval=2.0, rounds=15, x=4):
    loop = asyncio.get_running_loop()
    next_at = loop.time()
    clean_ticks = 0

    for n in range(1, rounds + 1):
        next_at += interval
        await asyncio.sleep(max(0, next_at - loop.time()))

        if clean_ticks >= x and not queue.empty():
            intercepted = []
            while not queue.empty():
                intercepted.append(queue.get_nowait())
            print(f"{loop.time():8.2f}  tick {n}  intercepts={intercepted}")
            clean_ticks = 0
        else:
            print(f"{loop.time():8.2f}  tick {n}")
            clean_ticks += 1


async def main():
    queue = asyncio.Queue()
    task = asyncio.create_task(interceptor(queue))
    try:
        await ticker(queue)
    # except asyncio.CancelledError:
    #     print("--- done ---")
    finally:
        task.cancel()
        await asyncio.gather(task, return_exceptions=True)
        print("done")

await main()   # asyncio.run(main()) in a script


In [ ]:
async def receiver_loop(queue):
    while True:
        await asyncio.sleep(random.uniform(0.3, 4.0))
        if random.random() < 0.5:
            await queue.put("new comment arrived")


async def sender_loop(queue, interval=2, min_len_without_interception=3):
    loop = asyncio.get_running_loop()
    next_at = loop.time()
    without_interception = min_len_without_interception
    while True:
        next_at += interval
        await asyncio.sleep(max(0, next_at - loop.time()))

        if without_interception < 1 and not queue.empty():
            cur = queue.get_nowait()
            print(f"{loop.time():8.2f} intercepted {queue.qsize()}")
            without_interception = min_len_without_interception
        else:
            print(f"{loop.time():8.2f}  tick")
            without_interception -= 1


async def main():
    queue = asyncio.Queue()
    task = asyncio.create_task(receiver_loop(queue))
    try:
        await sender_loop(queue)
    # except asyncio.CancelledError:
    #     print("--- done ---")
    finally:
        task.cancel()
        await asyncio.gather(task, return_exceptions=True)
        print("done")

await main()   # asyncio.run(main()) in a script


---

In [2]:
from app.db import seed_DB

DB = seed_DB()

In [6]:
DB

{4: ConversationStarter(ID=4, text='i think', language='en', timestamp=datetime.datetime(2026, 9, 14, 19, 28, 51, 595475), topics=(<Topic.what_is_democracy: 'what_is_democracy'>,)),
 5: Comment("hi"),
 6: Comment("bye")}

In [5]:
DB[5].topics

(<Topic.what_is_democracy: 'what_is_democracy'>,)

In [ ]:
async def receiver_loop(queue):
    while True:
        await asyncio.sleep(random.uniform(0.3, 4.0))
        if random.random() < 0.5:
            await queue.put("new comment arrived")


async def sender_loop(queue, interval=2, min_len_without_interception=3):
    loop = asyncio.get_running_loop()
    next_at = loop.time()
    without_interception = min_len_without_interception
    while True:
        next_at += interval
        await asyncio.sleep(max(0, next_at - loop.time()))

        if without_interception < 1 and not queue.empty():
            cur = queue.get_nowait()
            print(f"{loop.time():8.2f} intercepted {queue.qsize()}")
            without_interception = min_len_without_interception
        else:
            print(f"{loop.time():8.2f}  tick")
            without_interception -= 1


async def main():
    queue = asyncio.Queue()
    task = asyncio.create_task(receiver_loop(queue))
    try:
        await sender_loop(queue)
    # except asyncio.CancelledError:
    #     print("--- done ---")
    finally:
        task.cancel()
        await asyncio.gather(task, return_exceptions=True)
        print("done")

await main()   # asyncio.run(main()) in a script
